In [ ]:
data = {
    "student_id": [
        101, 102, 103, 104, 105,
        106, 107, 108, 109, 110,
        111, 112, 113, 114, 115,
        116, 117, 118, 119, 120,
        121, 122, 123, 124, 125,
        126, 127, 128, 129, 130,
        131, 132, 133, 134, 135,
        136, 137, 138, 139, 140
    ],

    "study_hours": [
        2, 6, 4, 9, 1,
        7, 5, 3, 8, 10,
        2, 4, 7, 6, 3,
        9, 5, 1, 8, 6,
        4, 10, 2, 7, 5,
        3, 9, 6, 8, 1,
        7, 4, 10, 2, 5,
        8, 3, 6, 9, 4
    ],

    "attendance": [
        68, 92, 75, 88, 54,
        81, 96, 63, 85, 97,
        72, 79, 91, 84, 66,
        93, 77, 58, 89, 73,
        86, 99, 61, 94, 78,
        69, 87, 95, 82, 57,
        90, 71, 98, 64, 80,
        76, 52, 83, 91, 70
    ],

    "assignments_completed": [
        5, 9, 6, 10, 2,
        8, 10, 5, 9, 10,
        4, 7, 9, 8, 5,
        10, 6, 3, 9, 7,
        8, 10, 4, 9, 6,
        5, 10, 8, 9, 2,
        9, 6, 10, 4, 7,
        8, 3, 9, 10, 5
    ],

    "previous_score": [
        48, 76, 61, 85, 32,
        72, 94, 51, 81, 91,
        45, 63, 79, 71, 55,
        88, 67, 39, 83, 59,
        74, 96, 44, 87, 69,
        53, 82, 90, 77, 35,
        86, 58, 93, 47, 64,
        73, 41, 80, 89, 56
    ],

    "sleep_hours": [
        6, 8, 5, 7, 4,
        7, 8, 6, 5, 8,
        6, 7, 9, 6, 5,
        8, 7, 4, 8, 6,
        7, 9, 5, 8, 6,
        5, 7, 8, 6, 4,
        7, 5, 8, 6, 7,
        9, 5, 7, 8, 6
    ],

    "screen_time": [
        6.8, 3.2, 5.5, 2.8, 9.0,
        4.1, 2.0, 7.5, 3.6, 2.5,
        8.2, 5.8, 2.9, 4.5, 7.0,
        2.2, 6.1, 9.3, 3.0, 5.2,
        4.0, 2.7, 8.5, 3.4, 6.3,
        7.8, 4.6, 2.4, 3.8, 8.9,
        3.1, 6.7, 2.1, 7.4, 5.0,
        4.3, 9.1, 3.5, 2.6, 6.0
    ],

    "final_score": [
        51, 79, 64, 87, 36,
        76, 95, 57, 83, 94,
        46, 67, 84, 74, 58,
        91, 70, 42, 88, 65,
        78, 97, 49, 90, 72,
        55, 86, 93, 81, 39,
        89, 61, 96, 52, 69,
        77, 45, 85, 92, 63
    ]
}

#---------------------------------------------------------------
# Pandas
#---------------------------------------------------------------

import pandas as pd, numpy as np
df = pd.DataFrame(data)

df[df['attendance']>80].mean()
df[df['attendance']<=80].mean()
df[(df['study_hours']<4) & (df['final_score']>70)][['study_hours','final_score']]
df[(df["attendance"]>85) & (df['screen_time']>6)][['attendance', 'screen_time']]


df['study_efficiency'] = df['study_hours']/(df['screen_time']+1)


assignments = df['assignments_completed']/(df['assignments_completed'].max())
attendance = df['attendance']/(df['attendance'].max())
study_hrs = df['study_hours']/(df['study_hours'].max())

df['academic_engagement'] = (assignments + attendance + study_hrs)/3

df['sleep_quality'] = df['sleep_hours']/7.5
df[['sleep_quality', 'final_score']]



#---------------------------------------------------------------
# NumPy
#---------------------------------------------------------------

X = df.drop('final_score', axis=1).to_numpy()
y = df['final_score'].to_numpy()



#---------------------------------------------------------------
# ML Pipeline
#---------------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor()
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

df_results = pd.DataFrame({
    "actual": y_test
})

for model_name, model in models.items():
  print(f"\nModel: {model_name}")
  pipeline = Pipeline(
      steps=[
          ('scaler', StandardScaler()),
          ('model', model)
      ]
  )
  pipeline.fit(X_train, y_train)
  y_pred = pipeline.predict(X_test)
  df_results[model_name] = y_pred - y_test
  mse = mean_squared_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)
  print(f"Root Mean Squared Error: {mse**(1/2)}")
  print(f"R-squared: {r2}")


df_results.sort_values('actual', ascending=False)



#-------------------------------------------------------
# PyTorch
#-------------------------------------------------------

import torch, torch.nn as nn, torch.optim as optim

x_data = torch.tensor(X, dtype=torch.float32)
y_data = torch.tensor(y, dtype=torch.float32)
y_data = y_data.reshape(-1, 1)

x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=42)

class Model(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(len(x_train[0]), 32)

    self.layer2 = nn.ReLU()

    self.layer3 = nn.Linear(32, 64)

    self.layer4 = nn.ReLU()

    self.layer5 = nn.Linear(64, 1)

  def forward(self, x):
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.layer5(x)
    return x


model = Model()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

def train_model(model, criterion, optimizer, x_train, y_train, epochs=1000, epoch_show=100):
  for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch+1) % epoch_show == 0:
      print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

print("\n\nPytorch Model...")

train_model(model, criterion, optimizer, x_train, y_train, epochs=10000, epoch_show=1000)


Model: Linear Regression
Root Mean Squared Error: 0.9249143386181179
R-squared: 0.9972793749671035

Model: Random Forest
Root Mean Squared Error: 2.996618928058753
R-squared: 0.9714419399721725


Pytorch Model...
Epoch: 1000, Loss: 1.2030950784683228
Epoch: 2000, Loss: 0.7180994749069214
Epoch: 3000, Loss: 1.4940764904022217
Epoch: 4000, Loss: 0.6368057131767273
Epoch: 5000, Loss: 0.49364516139030457
Epoch: 6000, Loss: 1.0658046007156372
Epoch: 7000, Loss: 0.26668286323547363
Epoch: 8000, Loss: 0.4847727119922638
Epoch: 9000, Loss: 0.198257178068161
Epoch: 10000, Loss: 0.36581486463546753


In [ ]:
model.eval()

with torch.no_grad():
    predictions = model(x_test)

mse = torch.mean((predictions - y_test) ** 2)
rmse = torch.sqrt(mse)

print("PyTorch RMSE:", rmse.item())

from sklearn.metrics import r2_score

r2 = r2_score(
    y_test.numpy(),
    predictions.numpy()
)

print("PyTorch R²:", r2)

PyTorch RMSE: 0.8665235042572021
PyTorch R²: 0.9976120591163635
